In [2]:
from pathlib import Path
import re
import pandas as pd

# Paths
BASE_DIR = Path.cwd()
BRANDS_PATH = BASE_DIR / "matched_phising_brands.csv"
URLS_PATH = BASE_DIR / "DevESS-AI-projectPhising_output.csv"

# Load brand names produced in stage-1
brands_raw_df = pd.read_csv(BRANDS_PATH)
brand_col = "BRAND_OR_URL" if "BRAND_OR_URL" in brands_raw_df.columns else brands_raw_df.columns[0]

brands_df = (
    brands_raw_df[[brand_col]]
    .rename(columns={brand_col: "brand"})
    .assign(brand=lambda d: d["brand"].astype(str).str.strip().str.lower())
)
brands_df = brands_df[brands_df["brand"].ne("")].drop_duplicates().reset_index(drop=True)

# Load phishing URLs (single-column file, no header)
all_phishing_urls_df = pd.read_csv(URLS_PATH, header=None, names=["url"])
all_phishing_urls_df["url"] = all_phishing_urls_df["url"].astype(str).str.strip()

#  Keep only URLs containing any known brand keyword
brand_keywords = sorted(brands_df["brand"].tolist(), key=len, reverse=True)
brand_pattern = "|".join(re.escape(b) for b in brand_keywords)

brand_matched_urls_df = all_phishing_urls_df[
    all_phishing_urls_df["url"].str.contains(brand_pattern, case=False, regex=True, na=False)
].copy()

# capture which brand matched first
brand_matched_urls_df["matched_brand"] = (
    brand_matched_urls_df["url"]
    .str.lower()
    .str.extract(f"({brand_pattern})", expand=False)
)

non_brand_urls_df = all_phishing_urls_df[
    ~all_phishing_urls_df.index.isin(brand_matched_urls_df.index)
].copy()

# 4) Persist intermediate CSVs for stage-2 workflow
brand_matched_urls_df.to_csv(BASE_DIR / "brand_matched_phishing_urls.csv", index=False)

print(f"Marka iceren URL: {len(brand_matched_urls_df):,}")

brand_matched_urls_df.head()

Marka iceren URL: 41,055


,url,matched_brand
20,gratisindirim.com.tr,gratis
30,dyson-kampanya.com,dyson
55,apple-ads-metric.com,apple
83,karacaucuzsatisadresi.com,karaca
105,meta-info-70349.herokuapp.com,meta


In [3]:
import numpy as np

# Cell 1 - Prepare RF training data
real_urls = [
    "akbank.com", "akbank.com.tr", "ziraatbank.com.tr", "ziraatbank.com", "garantibbva.com.tr", "yapikredi.com.tr",
    "isbank.com.tr", "halkbank.com.tr", "vakifbank.com.tr", "qnbfinansbank.com",
    "denizbank.com", "enpara.com", "kuveytturk.com.tr", "binance.com",
    "btcturk.com", "coinbase.com", "turkiye.gov.tr", "cimer.gov.tr",
    "turkcell.com.tr", "turktelekom.com.tr", "bimcell.com.tr", "sahibinden.com",
    "a101.com.tr", "yurticikargo.com", "pttavm.com", "arcelik.com.tr",
    "gratis.com", "karaca.com", "dyson.com.tr", "stanley1913.com",
    "hotic.com.tr", "adidas.com.tr", "amazon.com.tr", "apple.com",
    "icloud.com", "google.com", "youtube.com", "meta.com",
    "microsoft.com", "reddit.com", "anydesk.com", "robinhood.com", "booking.com", "vercel.com"
]

phishing_df = brand_matched_urls_df[["url"]].dropna().drop_duplicates().copy()
phishing_df["label"] = 1

benign_df = pd.DataFrame({"url": real_urls})
benign_df["url"] = benign_df["url"].astype(str).str.strip().str.lower()
benign_df = benign_df[benign_df["url"].ne("")].drop_duplicates().copy()
benign_df["label"] = 0

rf_train_df = pd.concat([phishing_df, benign_df], ignore_index=True)
rf_train_df["url"] = rf_train_df["url"].astype(str).str.strip().str.lower()
rf_train_df = rf_train_df[rf_train_df["url"].ne("")].drop_duplicates(subset=["url", "label"])

print("RF train veri dagilimi:")
print(rf_train_df["label"].value_counts().sort_index())
rf_train_df.head()

RF train veri dagilimi:
label
0       44
1    41053
Name: count, dtype: int64


,url,label
0,gratisindirim.com.tr,1
1,dyson-kampanya.com,1
2,apple-ads-metric.com,1
3,karacaucuzsatisadresi.com,1
4,meta-info-70349.herokuapp.com,1


In [4]:
# Cell 2 - RF feature extraction
phishing_keywords = [
    "login", "giris", "secure", "verify", "update", "hesap", "sifre", "odeme", "banka",
    "wallet", "promo", "kampanya", "firsat", "confirm", "reset", "account", "signin"
]
suspicious_tlds = [".xyz", ".top", ".site", ".online", ".shop", ".click", ".sbs", ".cfd", ".fun"]


def extract_features(url: str) -> dict:
    url = str(url).lower().strip()
    host = url.split("/")[0]

    features = {
        "url_len": len(url),
        "host_len": len(host),
        "dot_count": url.count("."),
        "dash_count": url.count("-"),
        "underscore_count": url.count("_"),
        "slash_count": url.count("/"),
        "question_count": url.count("?"),
        "equal_count": url.count("="),
        "digit_count": sum(ch.isdigit() for ch in url),
        "digit_ratio": (sum(ch.isdigit() for ch in url) / max(len(url), 1)),
        "has_ip_pattern": int(bool(re.search(r"\b\d{1,3}(?:\.\d{1,3}){3}\b", url))),
        "has_at_symbol": int("@" in url),
        "has_https_token": int("https" in url),
        "has_http_token": int("http" in url),
        "subdomain_depth": max(host.count(".") - 1, 0),
        "is_long_domain": int(len(host) >= 25),
        "has_com_tr": int(".com.tr" in url),
        "has_gov_tr": int(".gov.tr" in url),
    }

    for kw in phishing_keywords:
        features[f"kw_{kw}"] = int(kw in url)

    for tld in suspicious_tlds:
        features[f"tld_{tld.replace('.', '')}"] = int(url.endswith(tld))

    return features

X_rf = pd.DataFrame(rf_train_df["url"].apply(extract_features).tolist())
y_rf = rf_train_df["label"].astype(int)

print(f"RF feature matrix: {X_rf.shape}")
X_rf.head()

RF feature matrix: (41097, 44)


,url_len,host_len,dot_count,dash_count,underscore_count,slash_count,question_count,equal_count,digit_count,digit_ratio,...,kw_signin,tld_xyz,tld_top,tld_site,tld_online,tld_shop,tld_click,tld_sbs,tld_cfd,tld_fun
0,20,20,2,0,0,0,0,0,0,0.000000,...,0,0,0,0,0,0,0,0,0,0
1,18,18,1,1,0,0,0,0,0,0.000000,...,0,0,0,0,0,0,0,0,0,0
2,20,20,1,2,0,0,0,0,0,0.000000,...,0,0,0,0,0,0,0,0,0,0
3,25,25,1,0,0,0,0,0,0,0.000000,...,0,0,0,0,0,0,0,0,0,0
4,29,29,2,2,0,0,0,0,5,0.172414,...,0,0,0,0,0,0,0,0,0,0


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

# Cell 3 - Random Forest training
X_train, X_test, y_train, y_test = train_test_split(
    X_rf,
    y_rf,
    test_size=0.2,
    random_state=42,
    stratify=y_rf,
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

Confusion Matrix:
[[   8    1]
 [ 233 7978]]

Classification Report:
              precision    recall  f1-score   support

           0     0.0332    0.8889    0.0640         9
           1     0.9999    0.9716    0.9855      8211

    accuracy                         0.9715      8220
   macro avg     0.5165    0.9303    0.5248      8220
weighted avg     0.9988    0.9715    0.9845      8220

ROC-AUC: 0.9773


In [6]:
# Cell 4 - Preprocessing and LSTM sequence preparation
TENSORFLOW_AVAILABLE = True

try:
    import tensorflow as tf
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.preprocessing.text import Tokenizer
except Exception as exc:
    TENSORFLOW_AVAILABLE = False
    print("TensorFlow import edilemedi. LSTM hucreleri calismayabilir:", exc)

lstm_source_urls = (
    brand_matched_urls_df["url"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)
lstm_source_urls = lstm_source_urls[lstm_source_urls.ne("")].drop_duplicates().reset_index(drop=True)

sequence_length = 40
sequence_step = 2
max_sequence_samples = 120000

if TENSORFLOW_AVAILABLE:
    corpus_text = "\n".join(lstm_source_urls.tolist())

    char_tokenizer = Tokenizer(char_level=True, lower=True, filters="")
    char_tokenizer.fit_on_texts([corpus_text])

    encoded_text = char_tokenizer.texts_to_sequences([corpus_text])[0]
    vocab_size = len(char_tokenizer.word_index) + 1

    X_seq, y_seq = [], []
    upper_bound = max(len(encoded_text) - sequence_length, 0)

    for i in range(0, upper_bound, sequence_step):
        X_seq.append(encoded_text[i:i + sequence_length])
        y_seq.append(encoded_text[i + sequence_length])
        if len(X_seq) >= max_sequence_samples:
            break

    X_seq = np.array(X_seq, dtype=np.int32)
    y_seq = np.array(y_seq, dtype=np.int32)

    print(f"LSTM source URL sayisi: {len(lstm_source_urls):,}")
    print(f"Vocabulary size: {vocab_size}")
    print(f"Sequence matrix: {X_seq.shape}, Target: {y_seq.shape}")
else:
    char_tokenizer = None
    vocab_size = 0
    X_seq = np.array([])
    y_seq = np.array([])

LSTM source URL sayisi: 41,053
Vocabulary size: 48
Sequence matrix: (120000, 40), Target: (120000,)


In [7]:
# Cell 5 - Char-level LSTM training
lstm_model = None
lstm_history = None
lstm_ready = False

if TENSORFLOW_AVAILABLE and len(X_seq) > 1000:
    from sklearn.model_selection import train_test_split
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Dense, Embedding, LSTM
    from tensorflow.keras.models import Sequential

    X_train_seq, X_val_seq, y_train_seq, y_val_seq = train_test_split(
        X_seq,
        y_seq,
        test_size=0.1,
        random_state=42,
    )

    lstm_model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=64, input_length=sequence_length),
        LSTM(128, return_sequences=False),
        Dense(vocab_size, activation="softmax"),
    ])

    lstm_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
    )

    lstm_history = lstm_model.fit(
        X_train_seq,
        y_train_seq,
        validation_data=(X_val_seq, y_val_seq),
        epochs=8,
        batch_size=256,
        callbacks=[early_stop],
        verbose=1,
    )
    lstm_ready = True
else:
    print("LSTM egitimi atlandi: TensorFlow yok veya sequence sayisi yetersiz.")

print("LSTM hazir mi?", lstm_ready)

Epoch 1/8


D:\Dev\deneme\PhisingDetectorAI\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


422/422 ━━━━━━━━━━━━━━━━━━━━ 28s 62ms/step - accuracy: 0.2851 - loss: 2.6176 - val_accuracy: 0.4094 - val_loss: 2.1372
Epoch 2/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 35s 83ms/step - accuracy: 0.4751 - loss: 1.9256 - val_accuracy: 0.5232 - val_loss: 1.7607
Epoch 3/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 37s 88ms/step - accuracy: 0.5572 - loss: 1.6453 - val_accuracy: 0.5837 - val_loss: 1.5644
Epoch 4/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 37s 88ms/step - accuracy: 0.6038 - loss: 1.4815 - val_accuracy: 0.6204 - val_loss: 1.4423
Epoch 5/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 38s 90ms/step - accuracy: 0.6320 - loss: 1.3741 - val_accuracy: 0.6375 - val_loss: 1.3715
Epoch 6/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 37s 87ms/step - accuracy: 0.6531 - loss: 1.2970 - val_accuracy: 0.6521 - val_loss: 1.3129
Epoch 7/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 36s 85ms/step - accuracy: 0.6668 - loss: 1.2395 - val_accuracy: 0.6614 - val_loss: 1.2708
Epoch 8/8
422/422 ━━━━━━━━━━━━━━━━━━━━ 35s 83ms/step - accuracy: 0.6783 - loss: 1.1943 - val_accuracy: 0.66

In [8]:
# Cell 6 - Phising URL prediction
from urllib.parse import urlparse
import re
import numpy as np
import random
import pandas as pd

def sample_with_temperature(probabilities, temperature=1.0):
    probabilities = np.asarray(probabilities).astype("float64")
    probabilities = np.maximum(probabilities, 1e-9)
    probabilities = np.log(probabilities) / max(temperature, 1e-3)
    exp_probs = np.exp(probabilities)
    probabilities = exp_probs / np.sum(exp_probs)
    return np.random.choice(len(probabilities), p=probabilities)

def generate_guided_from_seed(brand: str, max_chars=12, temperature=0.70):
    """
    LSTM'den sadece 'orta kısmı' (firsat, kampanya vb.) üretmesini ister.
    Uzantıyı (.com vb.) sistem temiz bir şekilde kendisi ekler.
    """
    seed = f"{brand}-"
    generated = seed
    middle_part = ""

    for _ in range(max_chars):
        encoded_context = char_tokenizer.texts_to_sequences([generated[-sequence_length:]])[0]
        encoded_context = pad_sequences([encoded_context], maxlen=sequence_length, padding="pre")

        pred = lstm_model.predict(encoded_context, verbose=0)[0]
        next_idx = sample_with_temperature(pred, temperature=temperature)
        next_char = char_tokenizer.index_word.get(next_idx, "")

        # Don't add a dot if the next character is a digit or punctuation
        if not next_char or next_char == ".":
            break

        generated += next_char
        middle_part += next_char

    valid_tlds = ['.com', '.net', '.xyz', '.online', '.site', '.app', '.com.tr', '.shop']
    tld = random.choice(valid_tlds)

    middle_part = middle_part.strip("-")

    final_url = f"{brand}-{middle_part}{tld}"

    return final_url

def normalize_to_host(raw_url: str) -> str:
    s = str(raw_url).strip().lower()
    if not s:
        return ""
    if "://" not in s:
        s = "http://" + s
    try:
        host = urlparse(s).netloc
    except Exception:
        return ""

    if ":" in host:
        host = host.split(":")[0]
    return host.strip(".")

def count_brand_hits(host: str, brands) -> int:
    return sum(1 for b in brands if b and b in host)

DOMAIN_REGEX = re.compile(
    r"^(?=.{1,253}$)(?!-)(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,63}$"
)
NOISE_PATTERNS = [
    r"\bvercel\.app\b",
    r"\bduckdns\.org\b"
    r"\bddns\.net\b",
    r"\bhopto\.org\b",
    r"\bwebflow\b",
]
MAX_BRAND_COUNT = 1

def is_valid_candidate(host: str, brands) -> bool:
    if not host:
        return False
    if not DOMAIN_REGEX.match(host):
        return False
    if ".." in host or "--" in host:
        return False
    if len(host) < 6 or len(host) > 80:
        return False

    for pattern in NOISE_PATTERNS:
        if re.search(pattern, host):
            return False

    brand_hits = count_brand_hits(host, brands)
    # Disable brand checks if there are too many brand hits
    if brand_hits == 0 or brand_hits > MAX_BRAND_COUNT:
        return False

    # Too many digits can be a sign of noise or non-human-like generation
    digit_ratio = sum(ch.isdigit() for ch in host) / max(len(host), 1)
    if digit_ratio > 0.35:
        return False

    return True

seed_brands = brands_df["brand"].dropna().astype(str).str.strip().str.lower().drop_duplicates().tolist()
urls_per_seed = 10

generated_urls = []
if lstm_ready:
    for seed in seed_brands:
        for _ in range(urls_per_seed):
            candidate = generate_guided_from_seed(brand=seed, max_chars=12, temperature=0.70)
            if len(candidate) >= 6 and "." in candidate:
                generated_urls.append(candidate)

lstm_generated_candidates_df = pd.DataFrame({"url": sorted(set(generated_urls))})
candidate_df = lstm_generated_candidates_df.copy()
candidate_df["host"] = candidate_df["url"].apply(normalize_to_host)

brand_list = brands_df["brand"].dropna().astype(str).lower().str.strip().unique().tolist()

candidate_df["is_valid_domain"] = candidate_df["host"].apply(
    lambda h: bool(DOMAIN_REGEX.match(h)) if h else False
)
candidate_df["brand_hits"] = candidate_df["host"].apply(lambda h: count_brand_hits(h, brand_list))
candidate_df["is_valid_candidate"] = candidate_df["host"].apply(
    lambda h: is_valid_candidate(h, brand_list)
)

validated_input_df = (
    candidate_df[candidate_df["is_valid_candidate"]]
    .drop_duplicates(subset=["host"])
    .reset_index(drop=True)
)

print(f"Toplam uretilen aday URL: {len(candidate_df):,}")
print(f"Validator sonrasi kalan aday URL: {len(validated_input_df):,}")
validated_input_df.head()

Toplam uretilen aday URL: 437
Validator sonrasi kalan aday URL: 371


,url,host,is_valid_domain,brand_hits,is_valid_candidate
0,a101-a101-kampany.com.tr,a101-a101-kampany.com.tr,True,1,True
1,a101-idal.online,a101-idal.online,True,1,True
2,a101-ilisveris.xyz,a101-ilisveris.xyz,True,1,True
3,a101-indirim-sahi.net,a101-indirim-sahi.net,True,1,True
4,a101-inim.com,a101-inim.com,True,1,True


In [9]:
# Cell 7 - Last validation with Random Forest
rf_threshold = 0.98

if validated_input_df.empty:
    scored_candidates_df = pd.DataFrame(
        columns=["url", "host", "risk_score", "is_potential_phishing"]
    )
    validated_candidates_df = scored_candidates_df.copy()
else:
    X_generated_rf = pd.DataFrame(
        validated_input_df["host"].apply(extract_features).tolist()
    )
    risk_scores = rf_model.predict_proba(X_generated_rf)[:, 1]

    scored_candidates_df = validated_input_df.copy()
    scored_candidates_df["risk_score"] = risk_scores
    scored_candidates_df["is_potential_phishing"] = scored_candidates_df["risk_score"] >= rf_threshold

    validated_candidates_df = (
        scored_candidates_df[scored_candidates_df["is_potential_phishing"]]
        .sort_values("risk_score", ascending=False)
        .reset_index(drop=True)
    )

print(f"RF esik: {rf_threshold}")
print(f"RF skorlanan aday sayisi: {len(scored_candidates_df):,}")
print(f"Esigi gecen aday sayisi: {len(validated_candidates_df):,}")
validated_candidates_df[["host", "risk_score"]].head()

RF esik: 0.98
RF skorlanan aday sayisi: 371
Esigi gecen aday sayisi: 224


,host,risk_score
0,ziraatbankasi-vurcel.com,1.0
1,garantibbva-kackirsatler.app,1.0
2,gratis-buckanisvuru.xyz,1.0
3,toyota-issellerim.xyz,1.0
4,gratis-basvuru.shop,1.0


In [10]:

from pymongo import MongoClient

MONGO_URI = "mongodb+srv://<username>:<password>@<cluster-url>/?retryWrites=true&w=majority"
MONGO_DB = "phishing_ai"
MONGO_COLLECTION = "validated_candidates"

mongo_client = MongoClient(MONGO_URI)
mongo_db = mongo_client[MONGO_DB]
mongo_collection = mongo_db[MONGO_COLLECTION]

lstm_output_path = BASE_DIR / "lstm_generated_candidates.csv"
validator_output_path = BASE_DIR / "validator_filtered_candidates.csv"
validated_output_path = BASE_DIR / "rf_validated_phishing_candidates.csv"

lstm_generated_candidates_df.to_csv(lstm_output_path, index=False)
validated_input_df.to_csv(validator_output_path, index=False)

if not validated_candidates_df.empty:
    mongo_docs = validated_candidates_df.to_dict(orient="records")
    try:
        mongo_collection.insert_many(mongo_docs)
    except Exception as e:
        print(f"MongoDB'ye kayıt eklenirken hata oluştu: {e}")
else:
    print("MongoDB'ye eklenecek validasyonlu kayıt yok.")

total_generated = len(candidate_df)
total_validator_pass = len(validated_input_df)
total_rf_pass = len(validated_candidates_df)

valid_domain_rate = candidate_df["is_valid_domain"].mean() if total_generated else 0.0
single_or_double_brand_rate = (
    candidate_df["brand_hits"].between(1, 2).mean() if total_generated else 0.0
)
validator_pass_rate = candidate_df["is_valid_candidate"].mean() if total_generated else 0.0
rf_pass_rate = (total_rf_pass / max(total_validator_pass, 1)) if total_validator_pass else 0.0

print(f"LSTM cikti dosyasi: {lstm_output_path.name}")
print(f"Validator cikti dosyasi: {validator_output_path.name}")
print(f"RF dogrulanmis cikti dosyasi: {validated_output_path.name}")
print(f"Toplam uretilen aday: {total_generated:,}")
print(f"Validator sonrasi kalan: {total_validator_pass:,}")
print(f"RF esigini gecen aday: {total_rf_pass:,}")
print(f"valid_domain_rate: {valid_domain_rate:.2%}")
print(f"single_or_double_brand_rate: {single_or_double_brand_rate:.2%}")
print(f"validator_pass_rate: {validator_pass_rate:.2%}")
print(f"rf_pass_rate: {rf_pass_rate:.2%}")

LSTM cikti dosyasi: lstm_generated_candidates.csv
Validator cikti dosyasi: validator_filtered_candidates.csv
RF dogrulanmis cikti dosyasi: rf_validated_phishing_candidates.csv
Toplam uretilen aday: 437
Validator sonrasi kalan: 371
RF esigini gecen aday: 224
valid_domain_rate: 100.00%
single_or_double_brand_rate: 99.54%
validator_pass_rate: 84.90%
rf_pass_rate: 60.38%
